# M01 · Como funcionam os LLMs: prática

Neste notebook vais:

1. Comparar a tokenização de português e inglês em vários tokenizadores e estimar custos.
2. Implementar a atenção causal do zero em PyTorch.
3. Treinar um mini-GPT de caracteres com texto português.
4. Experimentar temperatura e top-p no teu próprio modelo.
5. Espreitar por dentro de um LLM real do Hugging Face.

**Requisitos:** `uv sync` na raiz do repositório. Corre bem em CPU; com GPU (ou no Colab) o treino da secção 3 fica muito mais rápido.

Lição de apoio: [licao.html](licao.html)

In [ ]:
import math, time, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(42)
print("device:", device)

## 1. Tokenização e custos

Vamos contar tokens do mesmo texto em português e em inglês com o tokenizador dos modelos GPT recentes (`o200k_base`) e com dois tokenizadores de modelos abertos.

In [ ]:
import tiktoken
from transformers import AutoTokenizer

textos = {
    "pt": "O desenvolvimento sustentável das cidades portuguesas depende de investimentos contínuos em transportes públicos e habitação acessível.",
    "en": "The sustainable development of Portuguese cities depends on continuous investment in public transport and affordable housing.",
}

tokenizadores = {
    "gpt (o200k_base)": tiktoken.get_encoding("o200k_base"),
    "Qwen2.5": AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct"),
    "GPT-2 (2019)": AutoTokenizer.from_pretrained("gpt2"),
}

def contar(tok, texto):
    return len(tok.encode(texto))

linhas = []
for nome, tok in tokenizadores.items():
    pt, en = contar(tok, textos["pt"]), contar(tok, textos["en"])
    linhas.append({"tokenizador": nome, "tokens_pt": pt, "tokens_en": en, "pt_vs_en": f"{pt / en - 1:+.0%}"})
pd.DataFrame(linhas)

Vê como uma palavra longa é partida em pedaços:

In [ ]:
enc = tokenizadores["gpt (o200k_base)"]
for palavra in ["desenvolvimento", "inconstitucionalissimamente", "Porto", " Porto", "2026", "123456789"]:
    ids = enc.encode(palavra)
    print(f"{palavra!r:32} {len(ids)} tokens -> {[enc.decode([i]) for i in ids]}")

**Exercício 1.1.** Escreve uma função `custo_eur(texto_entrada, tokens_saida, preco_in, preco_out)` que estima o custo de um pedido. Os preços são em dólares por milhão de tokens; usa uma taxa de câmbio fixa. Depois calcula quanto custa resumir 10 000 emails de 400 palavras com uma resposta de 80 tokens cada.

Os preços mudam com frequência: confirma-os na página de preços de cada fornecedor.

In [ ]:
USD_EUR = 0.92

def custo_eur(texto_entrada: str, tokens_saida: int, preco_in: float, preco_out: float, enc=enc) -> float:
    # TODO: conta os tokens de entrada com `enc` e devolve o custo em euros
    raise NotImplementedError

In [ ]:
# Solução
def custo_eur(texto_entrada, tokens_saida, preco_in, preco_out, enc=enc):
    t_in = len(enc.encode(texto_entrada))
    return (t_in * preco_in + tokens_saida * preco_out) / 1e6 * USD_EUR

email = ("Bom dia, envio em anexo a fatura referente ao mês de outubro. " * 40)[:2400]  # ~400 palavras
por_email = custo_eur(email, 80, preco_in=3.0, preco_out=15.0)
print(f"Por email: {por_email:.5f} €  |  10 000 emails: {por_email * 10_000:.2f} €")

## 2. Atenção causal do zero

Implementa `atencao(q, k, v)` com máscara causal. Formas: `(batch, T, d)`.

$$\text{Atenção}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V$$

In [ ]:
def atencao(q, k, v):
    # TODO:
    # 1. pontuações = q @ k^T / sqrt(d)
    # 2. máscara causal: posições futuras = -inf
    # 3. softmax na última dimensão
    # 4. devolve pesos @ v e os pesos
    raise NotImplementedError

In [ ]:
# Solução
def atencao(q, k, v):
    T, d = q.shape[-2], q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)
    mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=q.device), diagonal=1)
    scores = scores.masked_fill(mask, float("-inf"))
    pesos = F.softmax(scores, dim=-1)
    return pesos @ v, pesos

q = k = v = torch.randn(1, 5, 16)
saida, pesos = atencao(q, k, v)
print(pesos[0].round(decimals=2))

# Confirma contra a implementação otimizada do PyTorch
ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("igual ao PyTorch:", torch.allclose(saida, ref, atol=1e-5))

Repara que a matriz de pesos é triangular inferior: cada linha (token) só distribui atenção pelos tokens anteriores e por si próprio.

## 3. Mini-GPT de caracteres

Vamos treinar um transformer pequeno a prever o próximo caractere de *Os Lusíadas*. A arquitetura é a mesma de um LLM, só que com um vocabulário de caracteres e poucos milhões de parâmetros.

In [ ]:
URL = "https://www.gutenberg.org/cache/epub/3333/pg3333.txt"  # Os Lusíadas, Projeto Gutenberg
try:
    texto = urllib.request.urlopen(URL, timeout=30).read().decode("utf-8")
    ini = texto.find("As armas e os Barões assinalados")
    texto = texto[ini if ini > 0 else 0:]
    fim = texto.find("*** END OF")
    texto = texto[:fim if fim > 0 else None]
except Exception as e:
    print("Sem acesso ao Gutenberg, usa um ficheiro teu:", e)
    texto = open("texto.txt", encoding="utf-8").read()

print(len(texto), "caracteres")
print(texto[:300])

In [ ]:
chars = sorted(set(texto))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

dados = torch.tensor(encode(texto), dtype=torch.long)
n = int(0.9 * len(dados))
treino, val = dados[:n], dados[n:]
print("vocabulário:", len(chars), "caracteres")

In [ ]:
CFG = dict(block=128, d=192, heads=6, camadas=4, dropout=0.1, batch=64, lr=3e-4, passos=3000)

def batch(split):
    d = treino if split == "treino" else val
    ix = torch.randint(len(d) - CFG["block"] - 1, (CFG["batch"],))
    x = torch.stack([d[i:i + CFG["block"]] for i in ix])
    y = torch.stack([d[i + 1:i + CFG["block"] + 1] for i in ix])
    return x.to(device), y.to(device)

class Bloco(nn.Module):
    def __init__(self, d, heads, dropout):
        super().__init__()
        self.heads = heads
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d), nn.Dropout(dropout))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)
        # (B, heads, T, C/heads)
        q, k, v = [t.view(B, T, self.heads, C // self.heads).transpose(1, 2) for t in (q, k, v)]
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        a = a.transpose(1, 2).contiguous().view(B, T, C)
        x = x + self.drop(self.proj(a))      # ligação residual
        x = x + self.mlp(self.ln2(x))        # ligação residual
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab, block, d, heads, camadas, dropout, **_):
        super().__init__()
        self.block = block
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(block, d)
        self.blocos = nn.Sequential(*[Bloco(d, heads, dropout) for _ in range(camadas)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab)

    def forward(self, idx, alvo=None):
        T = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        logits = self.head(self.ln(self.blocos(x)))
        perda = None if alvo is None else F.cross_entropy(logits.view(-1, logits.size(-1)), alvo.view(-1))
        return logits, perda

    @torch.no_grad()
    def gerar(self, idx, n, temperatura=1.0, top_p=1.0):
        for _ in range(n):
            logits, _ = self(idx[:, -self.block:])
            logits = logits[:, -1, :] / temperatura
            probs = F.softmax(logits, dim=-1)
            if top_p < 1.0:
                ord_p, ord_i = probs.sort(descending=True)
                fora = ord_p.cumsum(-1) - ord_p > top_p
                ord_p[fora] = 0
                probs = torch.zeros_like(probs).scatter(-1, ord_i, ord_p)
                probs = probs / probs.sum(-1, keepdim=True)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

modelo = MiniGPT(len(chars), **CFG).to(device)
print(f"{sum(p.numel() for p in modelo.parameters()) / 1e6:.2f} M parâmetros")

Antes de treinar, a perda deve rondar $\ln(\text{vocab})$, porque o modelo escolhe ao acaso. Confirma:

In [ ]:
_, perda = modelo(*batch("treino"))
print(f"perda inicial {perda.item():.2f}  vs  ln(vocab) = {math.log(len(chars)):.2f}")

In [ ]:
opt = torch.optim.AdamW(modelo.parameters(), lr=CFG["lr"])
historico = []

@torch.no_grad()
def avaliar(n=50):
    modelo.eval()
    r = {s: np.mean([modelo(*batch(s))[1].item() for _ in range(n)]) for s in ("treino", "val")}
    modelo.train()
    return r

t0 = time.time()
for passo in range(CFG["passos"] + 1):
    if passo % 300 == 0:
        r = avaliar(); historico.append({"passo": passo, **r})
        print(f"passo {passo:5d}  treino {r['treino']:.3f}  val {r['val']:.3f}  ({time.time() - t0:.0f}s)")
    _, perda = modelo(*batch("treino"))
    opt.zero_grad(set_to_none=True)
    perda.backward()
    opt.step()

pd.DataFrame(historico).set_index("passo").plot(title="Perda (cross-entropy)");

In [ ]:
contexto = torch.tensor([encode("As armas e os ")], device=device)
print(decode(modelo.gerar(contexto, 400, temperatura=0.8)[0].tolist()))

## 4. Temperatura e top-p

**Exercício 4.1.** Gera 200 caracteres com temperaturas 0.3, 0.8 e 1.5, e depois com temperatura 1.0 e top-p 0.5. Descreve a diferença em duas frases. Mede a diversidade com a proporção de palavras únicas.

In [ ]:
def diversidade(s):
    p = s.split()
    return len(set(p)) / max(len(p), 1)

for t, p in [(0.3, 1.0), (0.8, 1.0), (1.5, 1.0), (1.0, 0.5)]:
    s = decode(modelo.gerar(contexto, 200, temperatura=t, top_p=p)[0].tolist())
    print(f"--- T={t} top_p={p}  diversidade={diversidade(s):.2f}\n{s}\n")

## 5. Por dentro de um LLM real

Vamos carregar o Qwen2.5-0.5B-Instruct (cerca de 1 GB) e ver os tokens mais prováveis para um contexto, tal como na demo da lição.

In [ ]:
from transformers import AutoModelForCausalLM

nome = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(nome)
llm = AutoModelForCausalLM.from_pretrained(nome, torch_dtype=torch.float32).to(device).eval()

print(llm.config.num_hidden_layers, "camadas |", llm.config.hidden_size, "dimensão |",
      llm.config.num_attention_heads, "cabeças de query |", llm.config.num_key_value_heads, "cabeças KV (GQA)")

In [ ]:
@torch.no_grad()
def proximos(texto, k=10):
    ids = tok(texto, return_tensors="pt").to(device)
    logits = llm(**ids).logits[0, -1]
    p = F.softmax(logits, dim=-1)
    top = p.topk(k)
    return pd.DataFrame({"token": [tok.decode([i]) for i in top.indices], "prob": top.values.cpu().numpy().round(3)})

proximos("O Porto é uma")

Agora com o formato de chat. O `apply_chat_template` acrescenta os tokens especiais que o modelo viu durante o SFT:

In [ ]:
msgs = [{"role": "user", "content": "Em que rio fica a cidade do Porto? Responde numa frase."}]
prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
print(prompt)
ids = tok(prompt, return_tensors="pt").to(device)
out = llm.generate(**ids, max_new_tokens=40, do_sample=False)
print(tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True))

**Exercício 5.1.** Usa `output_attentions=True` (carrega o modelo com `attn_implementation="eager"`) e desenha com matplotlib o mapa de atenção da última camada, média das cabeças, para a frase «O gato que o João adotou dorme no sofá». Para onde olha o token «dorme»?

**Exercício 5.2.** Mede o efeito do KV cache: gera 200 tokens com `use_cache=True` e `use_cache=False` e compara os tempos.

In [ ]:
# Exercício 5.2 (solução)
for cache in (True, False):
    t0 = time.time()
    llm.generate(**ids, max_new_tokens=200, do_sample=False, use_cache=cache, min_new_tokens=200)
    print(f"use_cache={cache}: {time.time() - t0:.1f}s")

## Use case: mini-GPT e explorador de tokens

O mini-GPT da secção 3 é a primeira metade do use case. A segunda está em `use-case/explorador_tokens.py`: uma ferramenta de linha de comandos que compara contagem de tokens e custo de um ficheiro em vários tokenizadores.

```bash
uv run python modulos/01-como-funcionam-os-llms/use-case/explorador_tokens.py contrato.txt --saida 500
```

**Desafio:** treina o mini-GPT com outro autor (Eça de Queirós ou Fernando Pessoa) e com tokenização BPE em vez de caracteres (usa `tokenizers` para treinar um BPE de 2000 tokens). Compara a qualidade do texto gerado com o mesmo número de passos.

## Resumo

- Tokenização determina custo, contexto e alguns erros estranhos dos modelos.
- A atenção é uma média ponderada de values, com pesos calculados por queries e keys, e uma máscara causal.
- Um GPT é embeddings, uma pilha de blocos (atenção, MLP, residuais, normalização) e uma cabeça linear.
- Temperatura e top-p controlam o compromisso entre previsibilidade e variedade.